In [ ]:
!pip install datasets torch torchvision scikit-learn pandas tqdm

In [ ]:
%%writefile config.py
# --- Content from config.py ---

# Centralized configuration for the HPFL simulation

CONFIG = {
    # 1. Dataset and Model Configuration
    'dataset': 'FEMNIST',
    'non_iid_scenario': 'non-iid-label', # or 'iid'
    'model_name': 'SimpleCNN',
    'max_samples_per_client': 500, # Max samples per client, or None for all

    # 2. Federated Learning Parameters
    'num_rounds': 50,          # Total number of global training rounds
    'local_epochs': 1,           # Number of local training epochs on each client
    'learning_rate': 0.01,
    'batch_size': 64,

    # 3. System Architecture
    'num_clients': 100,
    'num_uavs': 6,
    'clients_per_uav': 16, # num_clients / num_uavs
    'clients_to_select': 10, # Number of clients selected by each UAV per round (m)

    # 4. HPFL Specific Parameters
    # 4.1. Dynamic Client Selection (DCS) weights
    'dcs_weights': {
        'alpha': 0.25, # Communication quality
        'beta': 0.25,  # Computation capacity
        'gamma': 0.25, # Data significance
        'delta': 0.25, # Contribution (loss improvement)
    },
    
    # 4.2. Similarity-based Clustering
    'initial_clusters_k': 3, # Initial number of clusters
    'cluster_threshold': 0.90, # Similarity threshold for merging/splitting (future use)

    # 5. Simulation Mode
    # 'baseline_mode': None, # Options: 'Hierarchical_FedAvg', 'DCS_Only', 'Clustering_Only', 'HPFL'
    'baseline_mode': 'HPFL', # Set to the full proposed method by default

    # 6. Evaluation
    'eval_every': 5, # Evaluate personalized and global accuracy every N rounds

    # 7. Miscellaneous
    'seed': 42, # For reproducibility
    'results_dir': 'results/', # Directory to save metrics and plots
}

In [ ]:
%%writefile data.py
# --- Content from data.py ---

from collections import defaultdict
import random

from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision.transforms import Compose, ToTensor, Normalize
import torch

class FEMNISTDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image']
        label = sample['character']
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

def load_femnist_dataset(split='train'):
    """
    Loads the FEMNIST dataset from Hugging Face.
    """
    return load_dataset("flwrlabs/femnist", split=split)

def partition_data(dataset, num_clients, scenario='non-iid-label', max_samples_per_client=None, test_split_ratio=0.1):
    """
    Partitions the dataset for a number of clients, creating train/test splits for each.
    """
    # 1. Group indices by writer_id in a single pass
    print("   - Grouping data by writer...")
    writer_to_indices = defaultdict(list) # 이거 그냥 key 없으면 자동으로 에러 안내고 key 만들어주는 딕셔너리임. 
    # This assumes the dataset is a Hugging Face Dataset object
    for i, writer_id in enumerate(dataset['writer_id']):
        writer_to_indices[writer_id].append(i)
    
    writer_ids = sorted(list(writer_to_indices.keys()))
    print("writer_ids sample: ", writer_ids[:10])
    
    # 2. Distribute writer_ids to clients
    writers_per_client = len(writer_ids) // num_clients
    print("총 writer 수: ", len(writer_ids))
    print("writer per client: ", writers_per_client)
    client_datasets = []
    
    print(f"   - Assigning writers to {num_clients} clients...")
    for i in range(num_clients):
        client_indices = []
        start_writer_idx = i * writers_per_client
        end_writer_idx = (i + 1) * writers_per_client if i < num_clients - 1 else len(writer_ids)
        
        client_writer_ids = writer_ids[start_writer_idx:end_writer_idx]
        
        # 3. Gather indices for the client's assigned writers
        for writer_id in client_writer_ids:
            client_indices.extend(writer_to_indices[writer_id])
        
        # random.shuffle(client_indices) should not shuflle to make non iid

        if max_samples_per_client and len(client_indices) > max_samples_per_client:
            client_indices = client_indices[:max_samples_per_client]

        # Split indices into training and testing
        split_idx = int(len(client_indices) * (1 - test_split_ratio))
        train_indices = client_indices[:split_idx]
        test_indices = client_indices[split_idx:]

        # Create subsets for train and test
        train_subset = dataset.select(train_indices)
        test_subset = dataset.select(test_indices)
        
        transform = Compose([
            ToTensor(),
            Normalize((0.5,), (0.5,))
        ])
        
        train_dset = FEMNISTDataset(train_subset, transform=transform)
        test_dset = FEMNISTDataset(test_subset, transform=transform)
        
        client_datasets.append((train_dset, test_dset))
        
    return client_datasets

In [ ]:
%%writefile models.py
# --- Content from models.py ---

import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict

class PersonalizedModel(nn.Module):
    """Combines a shared backbone with a personalized head.
    The backbone is the SimpleCNN, and the head is a simple linear layer.
    """
    def __init__(self, shared_backbone, personalized_head):
        super(PersonalizedModel, self).__init__()
        self.backbone = shared_backbone
        self.head = personalized_head

    def forward(self, x):
        features = self.backbone(x) # The backbone should output features
        output = self.head(features)
        return output

# Redefine the backbone to separate feature extraction from classification
class CNNBackbone(nn.Module):
    def __init__(self):
        super(CNNBackbone, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=1)
        self.fc1 = nn.Linear(64 * 5 * 5, 512) # Feature layer

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = F.relu(self.fc1(x))
        return x

class PersonalizedHead(nn.Module):
    """A personalized classifier head."""
    def __init__(self, num_classes=62):
        super(PersonalizedHead, self).__init__()
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.fc2(x)

In [ ]:
%%writefile client.py
# --- Content from client.py ---

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import time
import numpy as np

from models import CNNBackbone, PersonalizedHead

class Client:
    """Represents a ground client in the HPFL simulation."""
    def __init__(self, client_id, train_dataset, test_dataset, compute_power=1.0, comm_quality=1.0, device='cpu'):
        self.client_id = client_id
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        
        self.train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
        self.test_dataloader = DataLoader(test_dataset, batch_size=128, num_workers=2, pin_memory=True)
        
        self.device = device

        # Hardware and network attributes
        self.compute_power = compute_power
        self.comm_quality = comm_quality
        
        # Data attributes
        self.data_significance = len(train_dataset)
        
        # Model components
        self.backbone = CNNBackbone().to(self.device)
        self.head = PersonalizedHead().to(self.device)
        
        # Training state
        self.last_loss = -1
        self.optimizer = optim.SGD(list(self.backbone.parameters()) + list(self.head.parameters()), lr=0.01)
        self.criterion = torch.nn.CrossEntropyLoss().to(self.device)

    def local_train(self, shared_state_dict, epochs, lr):
        self.backbone.load_state_dict(shared_state_dict)
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

        self.backbone.train()
        self.head.train()

        start_time = time.time()
        total_loss = 0.0
        num_batches = 0

        printed_batch_info = False # for debugging

        for epoch in range(epochs):
            for images, labels in self.train_dataloader:
                # --- Debug TEST CODE ---
                if not printed_batch_info and self.client_id == 0: #only print for client 0 once
                    print(f"\n[TEST] Client {self.client_id} (Epoch {epoch+1})")
                    print(f"  - DataLoader provided a batch of images with shape: {images.shape}")
                    print(f"  - DataLoader provided a batch of labels with shape: {labels.shape}")
                    printed_batch_info = True
                # --- END OF TEST CODE ---
                images, labels = images.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                
                # Forward pass
                features = self.backbone(images)
                outputs = self.head(features)
                
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                
                total_loss += loss.item()
                num_batches += 1

        end_time = time.time()
        training_time = (end_time - start_time) / self.compute_power # Simulate compute power

        # Update last_loss for contribution scoring
        if num_batches > 0:
            self.last_loss = total_loss / num_batches

        updated_backbone_state = self.backbone.state_dict()
        comm_cost_bytes = sum(p.numel() * p.element_size() for p in updated_backbone_state.values())

        return updated_backbone_state, training_time, comm_cost_bytes

    def compute_score(self, weights):
        """Computes the client's selection score based on multiple factors.
        Score S = alpha*q + beta*c + gamma*d + delta*g
        """
        alpha, beta, gamma, delta = weights['alpha'], weights['beta'], weights['gamma'], weights['delta']
        
        # For now, we use the raw values. Normalization should happen at the UAV level.
        q = self.comm_quality
        c = self.compute_power
        d = self.data_significance
        
        # Contribution score (g) - higher is better
        # Use inverse of loss. Add a small epsilon to avoid division by zero.
        g = 1.0 / (self.last_loss + 1e-6) if self.last_loss != -1 else 0
        
        score = (alpha * q) + (beta * c) + (gamma * d) + (delta * g)
        return score

    def get_head_params(self):
        return self.head.state_dict()

In [ ]:
%%writefile dcs.py
# --- Content from dcs.py ---

import numpy as np

def normalize_features(feature_values):
    """Normalizes a list of feature values to the [0, 1] range."""
    min_val = np.min(feature_values)
    max_val = np.max(feature_values)
    
    if max_val == min_val:
        return np.zeros_like(feature_values)
        
    return (feature_values - min_val) / (max_val - min_val)

def compute_scores(clients, weights):
    """Computes the DCS score for each client and returns a list of (client, score) tuples.

    The score is a weighted sum of normalized features:
    S = alpha*q + beta*c + gamma*d + delta*g
    
    Args:
        clients (list of Client): The clients to be scored.
        weights (dict): A dictionary with keys 'alpha', 'beta', 'gamma', 'delta'.

    Returns:
        list: A list of tuples, where each tuple is (client, score).
    """
    if not clients:
        return []

    alpha = weights.get('alpha', 0.25)
    beta = weights.get('beta', 0.25)
    gamma = weights.get('gamma', 0.25)
    delta = weights.get('delta', 0.25)

    # 1. Extract raw feature values from all clients
    comm_qualities = np.array([client.comm_quality for client in clients])
    compute_powers = np.array([client.compute_power for client in clients])
    data_significance = np.array([client.data_significance for client in clients])
    
    # Contribution score (g) is inverse of loss. Higher is better.
    # Add a small epsilon to avoid division by zero.
    contributions = np.array([1.0 / (client.last_loss + 1e-6) if client.last_loss != -1 else 0 for client in clients])

    # 2. Normalize each feature across the clients
    norm_q = normalize_features(comm_qualities)
    norm_c = normalize_features(compute_powers)
    norm_d = normalize_features(data_significance)
    norm_g = normalize_features(contributions)

    # 3. Compute the final weighted score for each client
    scores = (alpha * norm_q) + (beta * norm_c) + (gamma * norm_d) + (delta * norm_g)
    
    client_scores = list(zip(clients, scores))
    
    return client_scores

In [ ]:
%%writefile clustering.py
# --- Content from clustering.py ---

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

def model_to_vector(model_state):
    """Converts a model's state_dict to a flattened numpy vector."""
    # Ensure tensors are on the CPU and converted to numpy
    return np.concatenate([p.cpu().numpy().flatten() for p in model_state.values()])

def compute_similarity_matrix(vectors):
    """Computes the pairwise cosine similarity matrix for a list of vectors."""
    return cosine_similarity(vectors)

def cluster_assignment(model_vectors, num_clusters):
    """Assigns models to clusters using K-Means.

    Args:
        model_vectors (list of np.ndarray): The list of model vectors.
        num_clusters (int): The number of clusters to create.

    Returns:
        dict: A dictionary mapping cluster_id to a list of model indices in that cluster.
    """
    if not model_vectors or num_clusters <= 0:
        return {}

    # Stack vectors into a matrix for K-Means
    X = np.array(model_vectors)
    
    # If the number of samples is less than clusters, we can't form k clusters.
    # Assign each sample to its own cluster.
    if X.shape[0] < num_clusters:
        assignments = {i: [i] for i in range(X.shape[0])}
        return assignments

    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
    # Group indices by cluster label
    assignments = {i: [] for i in range(num_clusters)}
    for i, label in enumerate(labels):
        assignments[label].append(i)
        
    return assignments

In [ ]:
%%writefile uav.py
# --- Content from uav.py ---

import numpy as np
from collections import OrderedDict
import torch

from client import Client # Assuming client.py is in the same directory
from models import CNNBackbone

class UAV:
    """Represents a UAV that aggregates updates from a group of clients."""
    def __init__(self, uav_id, clients):
        self.uav_id = uav_id
        self.clients = clients
        self.model = CNNBackbone() # Each UAV maintains a model

    def select_clients(self, m, weights):
        """Selects the top m clients based on their DCS scores."""
        if m >= len(self.clients):
            return self.clients

        scores = [client.compute_score(weights) for client in self.clients]
        
        # Normalize scores before selection (optional but good practice)
        # For simplicity, we select top-m based on raw scores here.
        
        # Get indices of top m clients
        top_m_indices = np.argsort(scores)[-m:]
        selected_clients = [self.clients[i] for i in top_m_indices]
        
        return selected_clients

    def aggregate_updates(self, client_updates):
        """Aggregates model updates from selected clients using Federated Averaging (FedAvg).

        Args:
            client_updates (list of tuples): Each tuple contains (client_state_dict, num_samples).

        Returns:
            OrderedDict: The aggregated model state_dict.
        """
        if not client_updates:
            return self.model.state_dict() # Return current state

        total_samples = sum(num_samples for _, num_samples in client_updates)
        
        # Get the keys from the first model
        state_dicts = [sd for sd, _ in client_updates]
        keys = state_dicts[0].keys()
        
        avg_state_dict = OrderedDict()

        for key in keys:
            # Sum weighted tensors for the current key
            summed_tensor = torch.zeros_like(state_dicts[0][key]) # Init on CPU
            for sd, num_samples in client_updates:
                weight = num_samples / total_samples
                summed_tensor += sd[key] * weight
            
            avg_state_dict[key] = summed_tensor
            
        return avg_state_dict

    def get_model_state(self):
        """Returns the state dictionary of the UAV's model."""
        return self.model.state_dict()

    def set_model_state(self, state_dict):
        """Sets the state of the UAV's model."""
        self.model.load_state_dict(state_dict)

In [ ]:
%%writefile satellite.py
# --- Content from satellite.py ---

import numpy as np
from collections import OrderedDict
import torch

# We will need clustering utilities, which we assume will be in clustering.py
from clustering import model_to_vector, cluster_assignment
from models import CNNBackbone

class Satellite:
    """Represents the satellite responsible for global clustering and aggregation."""
    def __init__(self, num_clusters_k):
        self.num_clusters_k = num_clusters_k
        self.cluster_models = {} # {cluster_id: model_state_dict}

    def cluster_and_aggregate(self, uav_models):
        """Performs model clustering and aggregation for each cluster.

        Args:
            uav_models (dict): A dictionary mapping uav_id to the UAV's model state_dict.

        Returns:
            dict: A dictionary mapping cluster_id to the aggregated cluster model state_dict.
        """
        if not uav_models:
            return {}

        uav_ids = list(uav_models.keys())
        model_states = list(uav_models.values())

        # 1. Convert model states to vectors for clustering
        model_vectors = [model_to_vector(state) for state in model_states]

        # 2. Assign models to clusters
        # The clustering function should return a dictionary {cluster_id: [uav_indices]}
        cluster_assignments = cluster_assignment(model_vectors, self.num_clusters_k)
        
        # 3. Aggregate models within each cluster
        new_cluster_models = {}
        for cluster_id, uav_indices in cluster_assignments.items():
            if not uav_indices:
                continue
            
            cluster_uav_states = [model_states[i] for i in uav_indices]
            
            # Simple FedAvg aggregation within the cluster
            aggregated_state_dict = self._federated_averaging(cluster_uav_states)
            new_cluster_models[cluster_id] = aggregated_state_dict

        self.cluster_models = new_cluster_models
        return self.cluster_models, cluster_assignments

    def _federated_averaging(self, state_dicts):
        """Averages the parameters of multiple model state_dicts."""
        if not state_dicts:
            return None

        # Get the keys from the first model
        keys = state_dicts[0].keys()
        num_models = len(state_dicts)
        
        # Initialize a new state_dict to store the average
        avg_state_dict = OrderedDict()

        for key in keys:
            # Sum the tensors for the current key from all models
            summed_tensor = torch.stack([sd[key] for sd in state_dicts]).sum(0)
            avg_state_dict[key] = summed_tensor / num_models
            
        return avg_state_dict

In [ ]:
%%writefile metrics.py
# --- Content from metrics.py ---

import torch
from torch.utils.data import DataLoader, ConcatDataset, random_split
import numpy as np

def compute_personalized_accuracy(client, device, test_loader):
    """Evaluates the accuracy of a client's personalized model on their local test set."""
    client.backbone.eval()
    client.head.eval()
    
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            # Move the data batch to the device
            images, labels = images.to(device), labels.to(device)    

            features = client.backbone(images)
            outputs = client.head(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total if total > 0 else 0
    return accuracy

def compute_global_accuracy(cluster_model_state, clients, global_test_dataset, device):
    """Evaluates the accuracy of a global (cluster) model on a global test set."""
    # Create a temporary model to load the state
    from models import CNNBackbone, PersonalizedHead
    backbone = CNNBackbone().to(device)
    head = PersonalizedHead().to(device)
    
    # The global model only has a backbone
    backbone.load_state_dict(cluster_model_state)
    backbone.eval()
    head.eval()

    test_loader = DataLoader(
        global_test_dataset, 
        batch_size=128, 
        num_workers=2, 
        pin_memory=True
    )
    
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            features = backbone(images)
            outputs = head(features) # Use the generic head
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total if total > 0 else 0
    return accuracy

class MetricsLogger:
    def __init__(self):
        self.metrics = {
            'round': [], 'personalized_accuracy': [], 'global_accuracy': [],
            'loss': [], 'comm_cost': [], 'time_cost': []
        }

    def log_round(self, round_idx, personalized_acc, global_acc, avg_loss, comm_cost, time_cost):
        self.metrics['round'].append(round_idx)
        self.metrics['personalized_accuracy'].append(personalized_acc)
        self.metrics['global_accuracy'].append(global_acc)
        self.metrics['loss'].append(avg_loss)
        self.metrics['comm_cost'].append(comm_cost)
        self.metrics['time_cost'].append(time_cost)
        
        print(f"Round {round_idx:3d} | Pers. Acc: {personalized_acc:6.2f}% | "
              f"Global Acc: {global_acc:6.2f}% | Avg Loss: {avg_loss:.4f} | "
              f"Comm Cost: {comm_cost/1024:8.2f} KB | Time: {time_cost:.2f}s")

    def save_to_file(self, filename):
        import pandas as pd
        pd.DataFrame(self.metrics).to_csv(filename, index=False)

In [ ]:
%%writefile trainer.py
# --- Content from trainer.py ---

import os
import random
import numpy as np
import torch
from tqdm import tqdm
from collections import OrderedDict

from config import CONFIG
from data import load_femnist_dataset, partition_data
from client import Client
from uav import UAV
from satellite import Satellite
from dcs import compute_scores
from metrics import MetricsLogger, compute_personalized_accuracy, compute_global_accuracy
from models import CNNBackbone # Import the backbone model

def set_seed(seed):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def initialize_simulation(config):
    """Initializes the entire simulation environment."""
    print("1. Initializing simulation...")
    set_seed(config['seed'])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   - Using device: {device}")

    # Load and partition dataset
    print("   - Loading and partitioning dataset...")
    full_hf_dataset = load_femnist_dataset()
    
    # Split the Hugging Face dataset into training and testing sets
    hf_dataset_split = full_hf_dataset.train_test_split(test_size=0.1, seed=config['seed'])
    train_hf_dataset = hf_dataset_split['train']
    test_hf_dataset = hf_dataset_split['test']

    client_datasets = partition_data(
        train_hf_dataset, 
        config['num_clients'], 
        max_samples_per_client=config.get('max_samples_per_client')
    ) # split train dataset and test dataset for client.
    
    from data import FEMNISTDataset
    from torchvision.transforms import Compose, ToTensor, Normalize
    
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    global_test_dataset = FEMNISTDataset(test_hf_dataset, transform=transform)
    
    # Create clients
    print("   - Creating clients...")
    clients = []
    for i in range(config['num_clients']):
        train_d, test_d = client_datasets[i]
        compute_power = np.random.uniform(0.5, 1.5)
        comm_quality = np.random.uniform(0.5, 1.5)
        client = Client(client_id=i, train_dataset=train_d, test_dataset=test_d, compute_power=compute_power, comm_quality=comm_quality, device=device)
        clients.append(client)

    # Create UAVs and assign clients
    print("   - Creating UAVs and assigning clients...")
    uavs = []
    clients_per_uav = config['clients_per_uav']
    for i in range(config['num_uavs']):
        start_idx = i * clients_per_uav
        end_idx = (i + 1) * clients_per_uav
        uav_clients = clients[start_idx:end_idx]
        uav = UAV(uav_id=i, clients=uav_clients)
        uavs.append(uav)

    # Create Satellite
    print("   - Creating Satellite...")
    satellite = Satellite(num_clusters_k=config['initial_clusters_k'])

    # Initialize a global model on the satellite to start with
    initial_global_model = CNNBackbone().state_dict()
    satellite.cluster_models = {0: initial_global_model} # Start with one cluster

    print("Initialization complete.")
    return clients, uavs, satellite, global_test_dataset, device 

def run_experiment(config):
    """Runs the full HPFL experiment."""
    clients, uavs, satellite, global_test_dataset, device= initialize_simulation(config) # add device (gpu/cpu)
    logger = MetricsLogger()

    # Get baseline mode from config
    mode = config['baseline_mode']
    print(f"\n2. Starting experiment in mode: {mode}\n")

    for round_idx in range(1, config['num_rounds'] + 1):
        round_losses = []
        round_comm_costs = []
        round_time_costs = []
        uav_aggregated_models = {}

        # --- UAV and Client Level --- #
        for uav in tqdm(uavs, desc=f"Round {round_idx} - UAVs"):
            # Get the appropriate model for this UAV (based on previous round's clustering)
            # For simplicity, we can have a default model or more complex logic here.
            # In this version, we assume a single global model is broadcast to all.
            # A more advanced version would map UAVs to clusters.
            global_model_state = list(satellite.cluster_models.values())[0]

            # Client Selection
            if mode == 'Hierarchical_FedAvg' or mode == 'Clustering_Only':
                # Random selection
                selected_clients = random.sample(uav.clients, config['clients_to_select'])
            else: # 'DCS_Only' or 'HPFL'
                # DCS-based selection
                client_scores = compute_scores(uav.clients, config['dcs_weights'])
                selected_clients = [cs[0] for cs in sorted(client_scores, key=lambda x: x[1], reverse=True)[:config['clients_to_select']]]

            # Local Training
            client_updates = []
            max_train_time = 0
            for client in selected_clients:
                updated_state_dict, train_time, comm_cost = client.local_train(
                    global_model_state, config['local_epochs'], config['learning_rate']
                )
                client_updates.append((updated_state_dict, len(client.train_dataset)))
                round_losses.append(client.last_loss)
                round_comm_costs.append(comm_cost)
                if train_time > max_train_time:
                    max_train_time = train_time
            
            round_time_costs.append(max_train_time)

            # UAV Aggregation
            if client_updates:
                aggregated_state_dict = uav.aggregate_updates(client_updates)
                uav.model.load_state_dict(aggregated_state_dict)

            uav_aggregated_models[uav.uav_id] = uav.model.state_dict()

        # --- Satellite Level --- #
        if mode == 'Clustering_Only' or mode == 'HPFL':
            # Clustering and Global Aggregation
            cluster_models, _ = satellite.cluster_and_aggregate(uav_aggregated_models)
        else: # 'Hierarchical_FedAvg' or 'DCS_Only'
            # Simple global aggregation without clustering
            all_uav_states = list(uav_aggregated_models.values())
            global_model = satellite._federated_averaging(all_uav_states)
            cluster_models = {0: global_model}
        
        satellite.cluster_models = cluster_models

        # --- Evaluation --- #
        # no need to use gpu in uav, satellite . use in eval
        if round_idx % config['eval_every'] == 0:
            # Personalized Accuracy
            pers_accs = [compute_personalized_accuracy(c, device, c.test_dataloader) for c in clients]
            avg_pers_acc = np.mean(pers_accs)

            # Global Accuracy (evaluate each cluster model and average)
            global_accs = [compute_global_accuracy(cm, clients, global_test_dataset, device) for cm in cluster_models.values()]
            avg_global_acc = np.mean(global_accs)

            # Log metrics
            logger.log_round(
                round_idx=round_idx,
                personalized_acc=avg_pers_acc,
                global_acc=avg_global_acc,
                avg_loss=np.mean(round_losses),
                comm_cost=np.sum(round_comm_costs),
                time_cost=np.sum(round_time_costs) # Simplified: sum of max times per UAV zone
            )

    print("\n3. Experiment finished.")
    # Save results
    results_dir = config['results_dir']
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    results_file = os.path.join(results_dir, f"{mode}_metrics.csv")
    logger.save_to_file(results_file)
    print(f"Results saved to {results_file}")

In [ ]:


from trainer import run_experiment
from config import CONFIG
run_experiment(CONFIG)

